In [1]:
import os

os.environ['KAGGLE_USERNAME'] = 'niharikasaha'
os.environ['KAGGLE_KEY'] = 'KGAT_9e3d8e025b9d433c6bb5efc5605acecc'

!pip install kaggle --quiet

!kaggle datasets list -s daisee

ref                                         title                                        size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------  ------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
olgaparfenova/daisee                        DAiSEE                                15328963023  2023-12-25 20:59:34.487000           1922         12  0.375            
nidalshahin/daisee-frames                   DAiSEE Frames                          3844185376  2025-12-03 19:29:47.950000             18          0  0.47058824       
seversnape/daisee-videos                    daisee-videos                         15328963023  2026-02-28 10:25:21.343000              1          1  0.23529412       
nidalshahin/preprocessed-daisee-frames      Preprocessed DAiSEE Frames              920842829  2025-12-17 23:33:38.123000              6          0  0.3529412       

In [2]:
!kaggle datasets download -d olgaparfenova/daisee --force
!unzip -o daisee.zip -d daisee
!ls daisee

Streaming output truncated to the last 5000 lines.
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572051/2100572051.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572054/2100572054.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572055/2100572055.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572056/2100572056.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572057/2100572057.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572058/2100572058.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572059/2100572059.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572060/2100572060.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572061/2100572061.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572062/2100572062.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572063/2100572063.avi  
  inflating: daisee/DAiSEE/DataSet/Train/210057/2100572064/2100572064.avi  
  inflating: daisee/DAiSEE/DataSet/Tr

In [3]:
#load labels
import pandas as pd

train_df = pd.read_csv("daisee/DAiSEE/Labels/TrainLabels.csv")
train_df.head()

print(train_df.columns) #check

Index(['ClipID', 'Boredom', 'Engagement', 'Confusion', 'Frustration '], dtype='object')


In [4]:
#check if dataset properly loaded(just checking engagement for now)
train_df["label"] = train_df["Engagement"]

#convert to paths
import os

def get_video_path(clip_id):
    clip_base = clip_id.split('.')[0]
    user_id = clip_base[:6]

    base = f"daisee/DAiSEE/DataSet/Train/{user_id}/{clip_base}"

    avi_path = f"{base}/{clip_base}.avi"
    mp4_path = f"{base}/{clip_base}.mp4"

    if os.path.exists(avi_path):
        return avi_path
    elif os.path.exists(mp4_path):
        return mp4_path
    else:
        return None

train_df["video_path"] = train_df["ClipID"].apply(get_video_path)

# remove missing ones
train_df = train_df[train_df["video_path"].notna()].reset_index(drop=True)

#check
for i in range(10):
    p = train_df["video_path"].iloc[i]
    print(p, os.path.exists(p))

daisee/DAiSEE/DataSet/Train/110001/1100011002/1100011002.avi True
daisee/DAiSEE/DataSet/Train/110001/1100011003/1100011003.avi True
daisee/DAiSEE/DataSet/Train/110001/1100011004/1100011004.avi True
daisee/DAiSEE/DataSet/Train/110001/1100011005/1100011005.avi True
daisee/DAiSEE/DataSet/Train/110001/1100011006/1100011006.avi True
daisee/DAiSEE/DataSet/Train/110001/1100011007/1100011007.avi True
daisee/DAiSEE/DataSet/Train/110001/1100011008/1100011008.avi True
daisee/DAiSEE/DataSet/Train/110001/1100011009/1100011009.avi True
daisee/DAiSEE/DataSet/Train/110001/1100011010/1100011010.avi True
daisee/DAiSEE/DataSet/Train/110001/1100011011/1100011011.avi True


In [5]:
#load pretrained efficient net
import torch
import torch.nn as nn
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"

effnet = timm.create_model("efficientnet_b0", pretrained=True)
effnet.classifier = nn.Identity()
effnet = effnet.to(device)
effnet.eval()


import cv2
import numpy as np

def extract_frames(video_path, num_frames=30):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError(f"No frames in video: {video_path}")

    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    frames = []
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id in indices:
            frame = cv2.resize(frame, (224, 224))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        frame_id += 1

    cap.release()

    if len(frames) == 0:
        raise ValueError(f"No valid frames extracted: {video_path}")

    while len(frames) < num_frames:
        frames.append(frames[-1])

    return np.array(frames)


#extract_features
def extract_features(frames):
    frames = torch.tensor(frames).permute(0, 3, 1, 2).float() / 255.0
    frames = frames.to(device)

    with torch.no_grad():
        features = effnet(frames)  # ✅ use effnet, NOT model

    features = features.view(features.shape[0], -1)

    return features.cpu()

#test properly
path = train_df["video_path"].iloc[0]

frames = extract_frames(path)   # 🔥 ADD THIS LINE
features = extract_features(frames)

print(features.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

torch.Size([30, 1280])


In [6]:
#define model
import torch.nn as nn

class TemporalTransformer(nn.Module):
    def __init__(self, input_dim=1280):
        super().__init__()

        self.cls_token = nn.Parameter(torch.randn(1, 1, input_dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, 31, input_dim))  # 30 frames + CLS

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=8,
            dim_feedforward=1024,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 4)
        )

    def forward(self, x):
        B, T, D = x.shape

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embedding[:, :T+1, :]
        x = self.transformer(x)

        return self.mlp(x[:, 0])

#test model first(without dataset for now)
model = TemporalTransformer().to(device)

dummy = torch.randn(2, 30, 1280).to(device)
out = model(dummy)

print(out.shape)

torch.Size([2, 4])


In [7]:
#using a part of dataset (not full)
small_df = train_df.sample(50).reset_index(drop=True)

video_paths = small_df["video_path"].tolist()
labels = small_df["label"].values

In [8]:
#dataset class
from torch.utils.data import Dataset

class DAiSEEDataset(Dataset):
    def __init__(self, video_paths, labels):
        self.video_paths = video_paths
        self.labels = labels

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        frames = extract_frames(self.video_paths[idx])
        features = extract_features(frames)
        label = self.labels[idx]

        return features, torch.tensor(label, dtype=torch.long)

In [9]:
#dataloader
from torch.utils.data import DataLoader

dataset = DAiSEEDataset(video_paths, labels)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [10]:
#check
features, label = dataset[0]
print("Fixed shape:", features.shape)

Fixed shape: torch.Size([30, 1280])


In [11]:
for features, labels in loader:
    print("Batch shape:", features.shape)
    break

Batch shape: torch.Size([2, 30, 1280])


In [12]:
for features, labels in loader:
    print("BEFORE FIX:", features.shape)

    # 🔥 FIX SHAPE
    features = features.squeeze(1)  # remove extra dim if exists

    print("AFTER FIX:", features.shape)

    features = features.to(device)

    outputs = model(features)

    print("Outputs shape:", outputs.shape)
    break

BEFORE FIX: torch.Size([2, 30, 1280])
AFTER FIX: torch.Size([2, 30, 1280])
Outputs shape: torch.Size([2, 4])


In [13]:
#precompute features
from tqdm import tqdm

feature_dict = {}
skip_count = 0

for idx in tqdm(range(len(train_df))):
    path = train_df["video_path"].iloc[idx]

    try:
        frames = extract_frames(path)
        features = extract_features(frames)
        feature_dict[path] = features

    except Exception as e:
        skip_count += 1
        print("Skipping:", path)
        print("Reason:", str(e))

100%|██████████| 5358/5358 [29:25<00:00,  3.04it/s]


In [14]:
torch.save(feature_dict, "train_features.pt")

In [15]:
torch.save(feature_dict, "train_features.pt")

In [16]:
video_paths = list(feature_dict.keys())

# map labels
labels_map = dict(zip(train_df["video_path"], train_df["label"]))
labels = [labels_map[p] for p in video_paths]

In [17]:
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    video_paths, labels, test_size=0.2, random_state=42
)

In [18]:
class FastDAiSEEDataset(torch.utils.data.Dataset):
    def __init__(self, paths, labels, feature_dict):
        self.paths = paths
        self.labels = labels
        self.feature_dict = feature_dict

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        features = self.feature_dict[path]
        label = self.labels[idx]

        return features, torch.tensor(label, dtype=torch.long)

In [19]:
from torch.utils.data import DataLoader

train_dataset = FastDAiSEEDataset(train_paths, train_labels, feature_dict)
val_dataset = FastDAiSEEDataset(val_paths, val_labels, feature_dict)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [20]:
import torch.nn.functional as F

model = TemporalTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    train_loss = 0

    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)

        outputs = model(features)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch}, Train Loss: {train_loss/len(train_loader)}")

    # VALIDATION
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for features, labels in val_loader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"Validation Accuracy: {acc:.4f}")

Epoch 0, Train Loss: 0.880876106335156
Validation Accuracy: 0.5392
Epoch 1, Train Loss: 0.8426803877549385
Validation Accuracy: 0.6026
Epoch 2, Train Loss: 0.8274765728569743
Validation Accuracy: 0.5868
Epoch 3, Train Loss: 0.8187463855120674
Validation Accuracy: 0.5448
Epoch 4, Train Loss: 0.8052834853974741
Validation Accuracy: 0.5597
Epoch 5, Train Loss: 0.8315076723472395
Validation Accuracy: 0.6119
Epoch 6, Train Loss: 0.8347258634531676
Validation Accuracy: 0.4851
Epoch 7, Train Loss: 0.8634428550947958
Validation Accuracy: 0.4851
Epoch 8, Train Loss: 0.8661488619313311
Validation Accuracy: 0.4683
Epoch 9, Train Loss: 0.867279879860024
Validation Accuracy: 0.4888


In [21]:
torch.save(train_paths, "train_paths.pt")
torch.save(val_paths, "val_paths.pt")
torch.save(train_labels, "train_labels.pt")
torch.save(val_labels, "val_labels.pt")

In [22]:
from google.colab import files

files.download("train_features.pt")
files.download("train_paths.pt")
files.download("val_paths.pt")
files.download("train_labels.pt")
files.download("val_labels.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>